# [WIP] PointNet Packed

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean

import numpy as np

In [3]:
batchsize = 5

torch.from_numpy(np.array([1,0,0,0,1,0,0,0,1]).astype(np.float32)).view(1,9).repeat(batchsize,1)


tensor([[1., 0., 0., 0., 1., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1., 0., 0., 0., 1.]])

## Vanilla Implementation

In [4]:
class TNet(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        # Transformation network for input transformation
        self.k = k
        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k * k)
        
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)
        
    def forward(self, x, batch_index):
        """
        Args:
            x: (N, k) point cloud
            batch_index: (N,) batch assignments for each point
        Returns:
            (N, k, k) transformation matrix for each point
        """
        # Convert to (N, k, 1)
        x = x.unsqueeze(-1)
        
        # Point feature extraction
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        # Global feature extraction using scatter operations
        x = x.squeeze(-1)
        x = scatter_max(x, batch_index, dim=0)[0]
        
        # MLP for transformation matrix
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)
        
        # Initialize as identity
        iden = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x.view(-1, self.k, self.k) + iden
        
        # Broadcast transformation back to each point
        return x[batch_index]

In [5]:
class PointNet(nn.Module):
    def __init__(self, num_classes, input_channels=3, use_input_tnet=True, use_feature_tnet=True):
        super().__init__()
        self.use_input_tnet = use_input_tnet
        self.use_feature_tnet = use_feature_tnet
        
        if use_input_tnet:
            self.input_tnet = TNet(k=input_channels)
        if use_feature_tnet:
            self.feature_tnet = TNet(k=64)
        
        # Point feature extraction
        self.conv1 = nn.Conv1d(input_channels, 64, 1)
        self.conv2 = nn.Conv1d(64, 64, 1)
        self.conv3 = nn.Conv1d(64, 64, 1)
        self.conv4 = nn.Conv1d(64, 128, 1)
        self.conv5 = nn.Conv1d(128, 1024, 1)
        
        # Batch normalization layers
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(64)
        self.bn4 = nn.BatchNorm1d(128)
        self.bn5 = nn.BatchNorm1d(1024)
        
        # Classification head
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        
        self.bn6 = nn.BatchNorm1d(512)
        self.bn7 = nn.BatchNorm1d(256)
        
        self.dropout = nn.Dropout(p=0.3)
        
    def forward(self, x, batch_index):
        """
        Args:
            x: (N, C) packed point cloud data
            batch_index: (N,) batch assignments for each point
        Returns:
            (B, num_classes) classification logits
        """
        # Input transformation
        if self.use_input_tnet:
            trans = self.input_tnet(x, batch_index)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1)
        
        # Convert to (N, C, 1) for Conv1d
        x = x.unsqueeze(-1)
        
        # First MLP
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        # Feature transformation
        if self.use_feature_tnet:
            x = x.squeeze(-1)
            trans = self.feature_tnet(x, batch_index)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1).unsqueeze(-1)
        
        # Second MLP
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        
        # Global feature extraction
        x = x.squeeze(-1)
        x = scatter_max(x, batch_index, dim=0)[0]
        
        # Classification MLP
        x = F.relu(self.bn6(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn7(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        
        return x

In [6]:
# Example usage
def process_batch(points, batch_size):
    """
    Helper function to create batch indices for a packed tensor
    
    Args:
        points: (N, 3) packed point cloud
        batch_size: number of point clouds in the batch
    Returns:
        batch_index: (N,) tensor with batch assignments
    """
    device = points.device
    batch_index = torch.arange(batch_size, device=device).repeat_interleave(points.size(0) // batch_size)
    return batch_index

In [7]:
# Create model instance
model = PointNet(num_classes=40, input_channels=3)

# Example forward pass with packed format
points = torch.randn(1000, 3)  # 1000 points total (e.g., 500 + 500 from 2 point clouds)
batch_index = process_batch(points, batch_size=2)
output = model(points, batch_index)  # Shape: (2, 40)

In [8]:
output.shape

torch.Size([2, 40])

## Modular Refactoring

In [56]:
import sys

sys.path.append('..')

In [75]:
from torch_pointcloud.layers.convs import Conv1dBlock, LinearBlock


block = Conv1dBlock(
    in_channels=3,
    out_channels=64,
    kernel_size=1,
    stride=1,
    padding=0,
    # act=None,
    # dropout=None,
    # norm=None,
    # order="cand",
)

block

Conv1dBlock(
  (order): conv -> act -> norm -> dropout
  (conv): Conv1d(3, 64, kernel_size=(1,), stride=(1,), bias=False)
  (act): ReLU()
  (norm): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

In [76]:
x = torch.randn(1000, 3).unsqueeze(-1)
block(x).shape

torch.Size([1000, 64, 1])

In [81]:
block = LinearBlock(
    in_features=3,
    out_features=64,
    # act=None,
    dropout=None,
    # norm=None,
    order="andl",
)

block

LinearBlock(
  (order): act -> norm -> (dropout) -> linear
  (linear): Linear(in_features=3, out_features=64, bias=True)
  (act): ReLU()
  (norm): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [55]:
block = LinearBlock(
    in_features=64,
    out_features=128,
    act=None,
    dropout=None,
    norm=None,
    order="ldan",
)

block

LinearBlock(
  (order): linear -> (dropout) -> (act) -> (norm)
  (linear): Linear(in_features=64, out_features=128, bias=True)
)

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple, Union, Callable
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean

@dataclass
class PointNetConfig:
    """Configuration class for PointNet models"""
    input_channels: int = 3
    num_classes: int = 40
    embedding_dim: int = 1024
    use_input_tnet: bool = True
    use_feature_tnet: bool = True
    feature_transform_dim: int = 64
    dropout_rate: float = 0.3
    pool_method: str = 'max'  # 'max' or 'mean'
    stem_channels: Tuple[int, ...] = (64, 64, 64)
    head_channels: Tuple[int, ...] = (512, 256)
    use_batchnorm: bool = True

class MLPBlock(nn.Module):
    """Basic MLP block with optional BatchNorm and activation"""
    def __init__(
        self, 
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        use_batchnorm: bool = True,
        activation: Optional[Callable] = F.relu
    ):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size)
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        self.activation = activation

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return self.activation(x) if self.activation else x

class TNet(nn.Module):
    """Transformation Network"""
    def __init__(self, config: PointNetConfig, k: int):
        super().__init__()
        self.k = k
        
        # Feature extraction
        self.mlp1 = MLPBlock(k, 64, use_batchnorm=config.use_batchnorm)
        self.mlp2 = MLPBlock(64, 128, use_batchnorm=config.use_batchnorm)
        self.mlp3 = MLPBlock(128, 1024, use_batchnorm=config.use_batchnorm)
        
        # Transform estimation
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k * k)
        
        if config.use_batchnorm:
            self.bn4 = nn.BatchNorm1d(512)
            self.bn5 = nn.BatchNorm1d(256)
        else:
            self.bn4 = self.bn5 = nn.Identity()
        
        # Initialize last layer with zeros for identity transform
        nn.init.zeros_(self.fc3.weight)
        nn.init.eye_(self.fc3.bias.view(k, k))

    def forward(self, x, batch_index):
        x = x.unsqueeze(-1)
        
        # Feature extraction
        x = self.mlp1(x)
        x = self.mlp2(x)
        x = self.mlp3(x)
        
        # Global pooling and transform estimation
        x = x.squeeze(-1)
        x = scatter_max(x, batch_index, dim=0)[0]
        
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)
        
        # Add identity matrix
        iden = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x.view(-1, self.k, self.k) + iden
        
        return x[batch_index]

class PointNetEncoder(nn.Module):
    """PointNet encoder with configurable architecture"""
    def __init__(self, config: PointNetConfig):
        super().__init__()
        self.config = config
        
        # Input transform
        if config.use_input_tnet:
            self.input_tnet = TNet(config, config.input_channels)
        
        # Feature transform
        if config.use_feature_tnet:
            self.feature_tnet = TNet(config, config.feature_transform_dim)
        
        # Build stem MLP layers
        self.stem = nn.ModuleList()
        in_channels = config.input_channels
        for channels in config.stem_channels:
            self.stem.append(
                MLPBlock(in_channels, channels, use_batchnorm=config.use_batchnorm)
            )
            in_channels = channels
        
        # Global feature extraction
        self.global_feat = MLPBlock(
            in_channels, 
            config.embedding_dim, 
            use_batchnorm=config.use_batchnorm
        )

    def forward(self, x, batch_index):
        # Input transform
        if self.config.use_input_tnet:
            trans = self.input_tnet(x, batch_index)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1)
        
        x = x.unsqueeze(-1)
        
        # Stem forward
        for layer in self.stem:
            x = layer(x)
        
        # Feature transform
        if self.config.use_feature_tnet:
            x = x.squeeze(-1)
            trans = self.feature_tnet(x, batch_index)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1).unsqueeze(-1)
        
        # Global feature
        x = self.global_feat(x)
        x = x.squeeze(-1)
        
        # Global pooling
        if self.config.pool_method == 'max':
            x = scatter_max(x, batch_index, dim=0)[0]
        else:
            x = scatter_mean(x, batch_index, dim=0)
        
        return x

class PointNet(nn.Module):
    """Base PointNet model with configurable architecture"""
    def __init__(self, config: PointNetConfig):
        super().__init__()
        self.config = config
        
        # Encoder
        self.encoder = PointNetEncoder(config)
        
        # Classification head
        head_layers = []
        in_channels = config.embedding_dim
        
        for channels in config.head_channels:
            head_layers.extend([
                nn.Linear(in_channels, channels),
                nn.BatchNorm1d(channels) if config.use_batchnorm else nn.Identity(),
                nn.ReLU(),
                nn.Dropout(config.dropout_rate)
            ])
            in_channels = channels
        
        head_layers.append(nn.Linear(in_channels, config.num_classes))
        self.head = nn.Sequential(*head_layers)

    def forward(self, x, batch_index):
        x = self.encoder(x, batch_index)
        return self.head(x)

In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from typing import Optional, Tuple, Union, List, Callable

class MLPBlock(nn.Module):
    """Basic MLP block with BatchNorm and ReLU"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size)
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.bn(self.conv(x)))

class TNet(nn.Module):
    """T-Net for input and feature transformation"""
    def __init__(
        self,
        k: int,
        mlp_channels: Tuple[int, ...] = (64, 128, 1024),
        fc_channels: Tuple[int, ...] = (512, 256),
        use_batchnorm: bool = True,
    ):
        super().__init__()
        self.k = k
        
        # Feature extraction MLPs
        self.mlps = nn.ModuleList()
        in_channels = k
        for channels in mlp_channels:
            self.mlps.append(MLPBlock(in_channels, channels, use_batchnorm=use_batchnorm))
            in_channels = channels
        
        # FC layers
        self.fcs = nn.ModuleList()
        in_channels = mlp_channels[-1]
        for channels in fc_channels:
            self.fcs.append(nn.Linear(in_channels, channels))
            self.fcs.append(nn.BatchNorm1d(channels) if use_batchnorm else nn.Identity())
            in_channels = channels
        
        # Final transformation layer
        self.transform = nn.Linear(fc_channels[-1], k * k)
        
        # Initialize last layer for identity transform
        nn.init.zeros_(self.transform.weight)
        nn.init.eye_(self.transform.bias.view(k, k))

    def forward(self, x: torch.Tensor, batch_idxs: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1)
        
        # MLP feature extraction
        for mlp in self.mlps:
            x = mlp(x)
        
        # Global pooling
        x = x.squeeze(-1)
        x = scatter_max(x, batch_idxs, dim=0)[0]
        
        # FC layers
        for i in range(0, len(self.fcs), 2):
            x = F.relu(self.fcs[i+1](self.fcs[i](x)))
            
        # Transform
        x = self.transform(x)
        iden = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x.view(-1, self.k, self.k) + iden
        
        return x[batch_idxs]

class PointNetEncoder(nn.Module):
    """PointNet encoder with configurable architecture"""
    def __init__(
        self,
        input_channels: int = 3,
        stem_channels: Tuple[int, ...] = (64, 64, 64),
        head_channels: Tuple[int, ...] = (128, 1024),
        use_input_tnet: bool = True,
        use_feature_tnet: bool = True,
        use_batchnorm: bool = True,
        feature_transform_k: int = 64,
    ):
        super().__init__()
        self.use_input_tnet = use_input_tnet
        self.use_feature_tnet = use_feature_tnet
        
        # T-Nets
        if use_input_tnet:
            self.input_tnet = TNet(input_channels, use_batchnorm=use_batchnorm)
        if use_feature_tnet:
            self.feature_tnet = TNet(feature_transform_k, use_batchnorm=use_batchnorm)
        
        # Stem MLPs
        self.stem = nn.ModuleList()
        in_channels = input_channels
        for channels in stem_channels:
            self.stem.append(MLPBlock(in_channels, channels, use_batchnorm=use_batchnorm))
            in_channels = channels
            
        # Head MLPs
        self.head = nn.ModuleList()
        for channels in head_channels:
            self.head.append(MLPBlock(in_channels, channels, use_batchnorm=use_batchnorm))
            in_channels = channels
            
        self.out_channels = head_channels[-1]

    def forward(self, x: torch.Tensor, batch_idxs: torch.Tensor) -> torch.Tensor:
        # Input transform
        if self.use_input_tnet:
            trans = self.input_tnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1)
        
        # Stem
        x = x.unsqueeze(-1)
        for mlp in self.stem:
            x = mlp(x)
        
        # Feature transform
        if self.use_feature_tnet:
            x = x.squeeze(-1)
            trans = self.feature_tnet(x, batch_idxs)
            x = torch.bmm(x.unsqueeze(1), trans).squeeze(1).unsqueeze(-1)
        
        # Head
        for mlp in self.head:
            x = mlp(x)
        
        return x

class ClassificationHead(nn.Module):
    """Classification head with configurable architecture"""
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_channels: Tuple[int, ...] = (512, 256),
        dropout_rate: float = 0.3,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        layers = []
        for channels in hidden_channels:
            layers.extend([
                nn.Linear(in_channels, channels),
                nn.BatchNorm1d(channels) if use_batchnorm else nn.Identity(),
                nn.ReLU(True),
                nn.Dropout(dropout_rate)
            ])
            in_channels = channels
        
        layers.append(nn.Linear(in_channels, num_classes))
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

class PointNet(nn.Module):
    """Base PointNet model with configurable components"""
    def __init__(
        self,
        num_classes: int,
        encoder: PointNetEncoder,
        head: Optional[ClassificationHead] = None,
        global_pool: str = 'max',
    ):
        super().__init__()
        self.encoder = encoder
        self.head = head or ClassificationHead(
            encoder.out_channels,
            num_classes
        )
        self.global_pool = global_pool

    def forward(self, x: torch.Tensor, batch_idxs: torch.Tensor) -> torch.Tensor:
        # Feature extraction
        x = self.encoder(x, batch_idxs)
        
        # Global pooling
        x = x.squeeze(-1)
        if self.global_pool == 'max':
            x = scatter_max(x, batch_idxs, dim=0)[0]
        else:
            x = scatter_mean(x, batch_idxs, dim=0)
        
        # Classification
        return self.head(x)

# Model Variants
def pointnet_cls_base(num_classes: int = 40, **kwargs) -> PointNet:
    """Original PointNet classification model"""
    encoder = PointNetEncoder(
        input_channels=3,
        stem_channels=(64, 64, 64),
        head_channels=(128, 1024),
        use_input_tnet=True,
        use_feature_tnet=True
    )
    return PointNet(num_classes, encoder, **kwargs)

def pointnet_cls_small(num_classes: int = 40, **kwargs) -> PointNet:
    """Lightweight PointNet variant"""
    encoder = PointNetEncoder(
        input_channels=3,
        stem_channels=(32, 32, 64),
        head_channels=(128, 512),
        use_input_tnet=True,
        use_feature_tnet=False
    )
    head = ClassificationHead(
        512, num_classes,
        hidden_channels=(256, 128),
        dropout_rate=0.2
    )
    return PointNet(num_classes, encoder, head, **kwargs)

def pointnet_cls_tiny(num_classes: int = 40, **kwargs) -> PointNet:
    """Minimal PointNet variant"""
    encoder = PointNetEncoder(
        input_channels=3,
        stem_channels=(32, 64),
        head_channels=(128, 256),
        use_input_tnet=False,
        use_feature_tnet=False
    )
    head = ClassificationHead(
        256, num_classes,
        hidden_channels=(128,),
        dropout_rate=0.1
    )
    return PointNet(num_classes, encoder, head, **kwargs)

def pointnet_cls_large(num_classes: int = 40, **kwargs) -> PointNet:
    """Enhanced PointNet variant with larger capacity"""
    encoder = PointNetEncoder(
        input_channels=3,
        stem_channels=(64, 128, 128),
        head_channels=(256, 512, 2048),
        use_input_tnet=True,
        use_feature_tnet=True
    )
    head = ClassificationHead(
        2048, num_classes,
        hidden_channels=(1024, 512, 256),
        dropout_rate=0.4
    )
    return PointNet(num_classes, encoder, head, **kwargs)

In [86]:
# Create different model variants
models = {
    'base': pointnet_cls_base(40),
    'small': pointnet_cls_small(40),
    'tiny': pointnet_cls_tiny(40),
    'large': pointnet_cls_large(40)
}

# Test forward pass
x = torch.randn(1000, 3)
batch_idx = torch.zeros(1000, dtype=torch.long)

for name, model in models.items():
    model = model.eval()
    out = model(x, batch_idx)
    print(f"{name} model output shape: {out.shape}")

base model output shape: torch.Size([1, 40])
small model output shape: torch.Size([1, 40])
tiny model output shape: torch.Size([1, 40])


RuntimeError: Given groups=1, weight of size [64, 64, 1], expected input[1000, 128, 1] to have 64 channels, but got 128 channels instead